# Pixels to Predictions — SmolVLM Optimised (Colab)
**Runtime → Change runtime type → T4 GPU before running!**

## Step 1: Upload your competition zip from local machine

In [ ]:
# Run this cell — it will open a file picker to upload your zip/files
from google.colab import files
import os

print("Upload your competition data (zip file or individual CSVs + images folder).")
print("If it's a zip, upload the zip. If separate files, upload all of them.")
uploaded = files.upload()
print("\nUploaded:", list(uploaded.keys()))

## Step 2: Extract and locate data

In [ ]:
import zipfile, os, shutil

# Auto-extract any zip that was uploaded
for fname in uploaded.keys():
    if fname.endswith(".zip"):
        print(f"Extracting {fname}...")
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall("./competition_data")
        print("Extracted to ./competition_data")
        break
else:
    # No zip — move uploaded files into competition_data
    os.makedirs("./competition_data", exist_ok=True)
    for fname in uploaded.keys():
        shutil.move(fname, f"./competition_data/{fname}")
    print("Moved files to ./competition_data")

# Find train.csv
DATA_ROOT = None
for root, dirs, files_list in os.walk("."):
    for f in files_list:
        if f == "train.csv":
            DATA_ROOT = root
            print(f"Found train.csv at: {root}")
            break
    if DATA_ROOT: break

if DATA_ROOT is None:
    # Show what we have so user can debug
    print("Could not find train.csv. Directory structure:")
    for root, dirs, files_list in os.walk("./competition_data"):
        depth = root.replace("./competition_data", "").count(os.sep)
        print("  " * depth + os.path.basename(root) + "/")
        for f in files_list[:3]:
            print("  " * (depth+1) + f)
    raise FileNotFoundError("Upload the data again — train.csv not found.")

print(f"DATA_ROOT = {DATA_ROOT}")

## Step 3: Install dependencies

In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip("transformers>=4.46.0")
pip("peft>=0.13.0")
pip("accelerate>=0.34.0")

import torch, transformers, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — change runtime!")
assert torch.cuda.is_available(), "Go to Runtime > Change runtime type > T4 GPU"


## Step 4: Configuration

In [ ]:
import os, sys, json, math, random, gc, time
import numpy as np

OUTPUT_DIR = "/content"   # submission.csv will save here

# --- Model ----------------------------------------------------------------
MODEL_ID     = "HuggingFaceTB/SmolVLM-500M-Instruct"
MAX_IMG_SIZE = 448

# --- Training -------------------------------------------------------------
DO_TRAIN         = True
NUM_EPOCHS       = 3
BATCH_SIZE       = 1
GRAD_ACCUM_STEPS = 16       # effective batch 16
LR               = 5e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.10
LABEL_SMOOTHING  = 0.1
MAX_TRAIN_SAMPLES = None
SEED             = 42
TRAIN_ON_FULL    = True     # train on train+val for best score

# --- LoRA -----------------------------------------------------------------
LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "gate_proj", "up_proj", "down_proj"]

# --- Inference ------------------------------------------------------------
EVAL_BATCH_SIZE   = 1
EVAL_VAL          = not TRAIN_ON_FULL
N_ENSEMBLE_PASSES = 3
MAX_TEXT_CHARS    = 2000

random.seed(SEED)
np.random.seed(SEED)
import torch; torch.manual_seed(SEED)
print("Config ready. DATA_ROOT =", DATA_ROOT)

## Step 5: Load CSVs

In [ ]:
import pandas as pd, json, math

train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_ROOT, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))
print(f"train={len(train_df)} | val={len(val_df)} | test={len(test_df)}")

def parse_choices(s):
    if isinstance(s, list): return s
    try:    return json.loads(s)
    except: return [c.strip() for c in str(s).strip("[]").split(",")]

def clean(t, maxlen=MAX_TEXT_CHARS):
    if t is None or (isinstance(t, float) and math.isnan(t)): return ""
    t = str(t).strip()
    return t[:maxlen] + " ..." if len(t) > maxlen else t

for df in (train_df, val_df, test_df):
    df["choices_list"] = df["choices"].apply(parse_choices)
    df["num_choices"]  = df["choices_list"].apply(len)
    df["hint_c"]       = df["hint"].apply(clean)
    df["lecture_c"]    = df["lecture"].apply(clean)
    df["subject_c"]    = df["subject"].apply(clean) if "subject" in df.columns else ""

# Resolve image paths
def resolve_image_root(data_root):
    for candidate in [
        os.path.join(data_root, "images", "train"),
        os.path.join(data_root, "images", "images", "train"),
    ]:
        if os.path.exists(candidate):
            return data_root if "images/train" in candidate else os.path.join(data_root, "images")
    # fallback: search for any train image directory
    for root, dirs, files_l in os.walk(data_root):
        if "train" in os.path.basename(root) and any(f.endswith(".png") or f.endswith(".jpg") for f in files_l):
            return os.path.dirname(os.path.dirname(root))
    raise FileNotFoundError("Cannot locate image directories under " + data_root)

IMAGE_PREFIX = resolve_image_root(DATA_ROOT)

def resolve_path(rel_path):
    p = os.path.join(IMAGE_PREFIX, rel_path)
    if os.path.exists(p): return p
    p2 = os.path.join(IMAGE_PREFIX, rel_path.split("/", 1)[-1])
    return p2 if os.path.exists(p2) else None

for df in (train_df, val_df, test_df):
    df["image_full"] = df["image_path"].apply(resolve_path)

missing = [df["image_full"].isna().sum() for df in (train_df, val_df, test_df)]
print("Missing images train/val/test:", missing)
assert sum(missing) == 0, "Some images missing — check your zip structure"

if TRAIN_ON_FULL:
    full_train_df = pd.concat([train_df, val_df], ignore_index=True)
    print(f"Training on train+val: {len(full_train_df)} samples")
else:
    full_train_df = train_df
    print(f"Training on train only: {len(full_train_df)} samples")

## Step 6: Load model

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq

device = "cuda"
dtype  = torch.float16

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    _attn_implementation="eager",
).to(device)

tokenizer = processor.tokenizer
LETTERS   = ["A", "B", "C", "D", "E"]

def letter_token_id(letter):
    ids = tokenizer.encode(f" {letter}", add_special_tokens=False)
    return ids[-1]

LETTER_IDS = [letter_token_id(l) for l in LETTERS]
print("Letter IDs:", dict(zip(LETTERS, LETTER_IDS)))
assert len(set(LETTER_IDS)) == len(LETTER_IDS)
print(f"Model loaded — {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

## Step 7: Attach LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

for p in model.parameters():
    p.requires_grad = False

if DO_TRAIN:
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", target_modules=LORA_TARGETS, task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable LoRA params: {trainable:,} ({trainable/1e6:.3f}M)")
    assert trainable < 5_000_000, f"Exceeds 5M cap: {trainable:,}"


## Step 8: Dataset & prompts

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

train_aug = T.Compose([
    T.RandomResizedCrop(size=MAX_IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
])

def build_user_text(row):
    parts = [f"Question: {str(row['question']).strip()}"]
    if row.get("subject_c", ""): parts.append(f"Subject: {row['subject_c']}")
    if row.get("hint_c", ""):    parts.append(f"Hint: {row['hint_c']}")
    if row.get("lecture_c", ""): parts.append(f"Lecture: {row['lecture_c']}")
    parts.append("Choices:")
    for i, c in enumerate(row["choices_list"]):
        parts.append(f"{LETTERS[i]}. {str(c).strip()}")
    parts.append("Answer with ONLY the single letter (A/B/C/D/E) of the correct choice.")
    return "\n".join(parts)

def load_image(path, augment=False):
    img = Image.open(path).convert("RGB")
    if augment:
        return train_aug(img)
    w, h = img.size
    if max(w, h) > MAX_IMG_SIZE:
        scale = MAX_IMG_SIZE / max(w, h)
        img = img.resize((int(w*scale), int(h*scale)), Image.BILINEAR)
    return img

def build_chat_messages(row):
    return [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": build_user_text(row)},
    ]}]

class MCQADataset(Dataset):
    def __init__(self, df, with_labels=True, augment=False):
        self.df = df.reset_index(drop=True)
        self.with_labels = with_labels and "answer" in df.columns
        self.augment = augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row["image_full"], augment=self.augment)
        msgs  = build_chat_messages(row)
        prompt_text = processor.apply_chat_template(msgs, add_generation_prompt=True)
        item = {"image": image, "prompt_text": prompt_text, "row_idx": idx,
                "num_choices": int(row["num_choices"])}
        if self.with_labels:
            ans = int(row["answer"])
            item["answer_idx"] = ans
            item["letter_id"]  = LETTER_IDS[ans]
        return item

def collate_train(batch):
    texts  = [b["prompt_text"] + LETTERS[b["answer_idx"]] for b in batch]
    images = [[b["image"]] for b in batch]
    enc    = processor(text=texts, images=images, return_tensors="pt", padding=True)
    pad_id = tokenizer.pad_token_id or 0
    labels = torch.full_like(enc["input_ids"], -100)
    for i in range(enc["input_ids"].size(0)):
        nonpad = (enc["input_ids"][i] != pad_id).nonzero(as_tuple=True)[0]
        last   = nonpad[-1].item()
        labels[i, last] = enc["input_ids"][i, last]
    enc["labels"] = labels
    return enc

def collate_eval(batch):
    enc = processor(text=[b["prompt_text"] for b in batch],
                    images=[[b["image"]] for b in batch],
                    return_tensors="pt", padding=True)
    enc["_num_choices"] = torch.tensor([b["num_choices"] for b in batch], dtype=torch.long)
    enc["_row_idx"]     = torch.tensor([b["row_idx"]     for b in batch], dtype=torch.long)
    return enc

train_ds = MCQADataset(full_train_df, with_labels=True,  augment=True)
val_ds   = MCQADataset(val_df,        with_labels=True,  augment=False)
test_ds  = MCQADataset(test_df,       with_labels=False, augment=False)
print(f"train_ds={len(train_ds)} | val_ds={len(val_ds)} | test_ds={len(test_ds)}")

## Step 9: Training

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import torch.nn.functional as F

def smooth_loss(logits, labels, smoothing=LABEL_SMOOTHING):
    mask = labels != -100
    if not mask.any(): return logits.new_tensor(0.0)
    log_probs = F.log_softmax(logits[mask], dim=-1)
    targets   = labels[mask]
    nll       = -log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    smooth_v  = -log_probs.mean(dim=-1)
    return ((1 - smoothing) * nll + smoothing * smooth_v).mean()

def train_one_epoch():
    model.train()
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate_train, num_workers=2, pin_memory=True)
    n_opt_steps  = math.ceil(len(loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * n_opt_steps)
    optim  = AdamW([p for p in model.parameters() if p.requires_grad],
                   lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = get_cosine_schedule_with_warmup(optim, warmup_steps, n_opt_steps)
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(NUM_EPOCHS):
        running = 0.0; optim.zero_grad(); t0 = time.time()
        for i, batch in enumerate(loader):
            batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
                     for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.amp.autocast("cuda", dtype=dtype):
                out  = model(**batch, labels=labels)
                loss = smooth_loss(out.logits, labels) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            running += loss.item() * GRAD_ACCUM_STEPS

            if (i+1) % GRAD_ACCUM_STEPS == 0 or (i+1) == len(loader):
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 0.5)
                scaler.step(optim); scaler.update()
                sched.step(); optim.zero_grad()
                step = (i+1) // GRAD_ACCUM_STEPS
                if step % 25 == 0:
                    print(f"epoch {epoch+1} step {step} | loss {running/(i+1):.4f} "
                          f"| lr {sched.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
        print(f"=== epoch {epoch+1} done | avg loss {running/len(loader):.4f} ===")

if DO_TRAIN:
    train_one_epoch()
    gc.collect(); torch.cuda.empty_cache()


## Step 10: Inference (multi-pass ensemble)

In [ ]:
@torch.no_grad()
def predict_indices(dataset, batch_size=EVAL_BATCH_SIZE, desc="predict", n_passes=N_ENSEMBLE_PASSES):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=collate_eval, num_workers=2, pin_memory=True)
    preds      = [None] * len(dataset)
    pad_id     = tokenizer.pad_token_id or 0
    letter_t   = torch.tensor(LETTER_IDS, device=device)
    t0         = time.time(); n_done = 0

    for batch in loader:
        num_choices = batch.pop("_num_choices")
        row_idx     = batch.pop("_row_idx")
        batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
                 for k, v in batch.items()}
        B     = batch["input_ids"].size(0)
        accum = torch.zeros(B, 5, device=device)

        for pass_i in range(n_passes):
            model.train() if pass_i < n_passes-1 else model.eval()
            with torch.amp.autocast("cuda", dtype=dtype):
                out = model(**batch)
            for b in range(B):
                nonpad = (batch["input_ids"][b] != pad_id).nonzero(as_tuple=True)[0]
                lp = torch.log_softmax(out.logits[b, nonpad[-1].item()], dim=-1)
                accum[b] += lp[letter_t]

        for b in range(B):
            nc   = int(num_choices[b].item())
            pred = int(accum[b, :nc].argmax().item())
            preds[int(row_idx[b].item())] = pred

        n_done += B
        if n_done % 100 == 0:
            print(f"  {desc}: {n_done}/{len(dataset)} ({time.time()-t0:.0f}s)")

    model.eval()
    return preds

## Step 11: Validation accuracy

In [ ]:
if EVAL_VAL:
    val_preds = predict_indices(val_ds, desc="val")
    y_true = val_df["answer"].astype(int).tolist()
    acc = float(np.mean([p == t for p, t in zip(val_preds, y_true)]))
    print(f"Validation accuracy: {acc:.4f}  ({sum(p==t for p,t in zip(val_preds,y_true))}/{len(y_true)})")
else:
    print("Skipped (trained on full train+val).")

## Step 12: Generate submission.csv

In [ ]:
test_preds = predict_indices(test_ds, desc="test")

submission = pd.DataFrame({
    "id":     test_df["id"].tolist(),
    "answer": [int(p) for p in test_preds],
})

# Clamp any out-of-range predictions
for i, (a, k) in enumerate(zip(submission["answer"], test_df["num_choices"].astype(int))):
    if not (0 <= a < k):
        submission.at[i, "answer"] = 0

sample_sub = os.path.join(DATA_ROOT, "sample_submission.csv")
if os.path.exists(sample_sub):
    samp = pd.read_csv(sample_sub)
    submission = submission.set_index("id").loc[samp["id"]].reset_index()

out_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(submission.head())
print("Distribution:", submission["answer"].value_counts().sort_index().to_dict())

## Step 13: Download submission.csv to your computer

In [ ]:
from google.colab import files
files.download("/content/submission.csv")
print("Download started!")

## Step 14: (Optional) Save LoRA adapter to Google Drive

In [ ]:
# Uncomment to mount Drive and save the adapter
# from google.colab import drive
# drive.mount('/content/drive')
# save_path = "/content/drive/MyDrive/smolvlm_lora_adapter"
# model.save_pretrained(save_path)
# processor.save_pretrained(save_path)
# print("Saved adapter to Drive:", save_path)

# Or just download the adapter folder as a zip:
if DO_TRAIN:
    import shutil
    adapter_dir = "/content/lora_adapter"
    model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)
    shutil.make_archive("/content/lora_adapter", "zip", adapter_dir)
    files.download("/content/lora_adapter.zip")
    print("Adapter zip download started.")